# Segmentacion con Inteligencia Artificial

Este notebook cubre la segmentacion de imagenes desde sus fundamentos hasta implementaciones practicas en PyTorch. Se trabajan las arquitecturas fundamentales (FCN, U-Net), todas las funciones de perdida relevantes, las metricas estandar del campo y la augmentacion sincronizada imagen-mascara.

Para garantizar reproducibilidad sin dependencias de descarga, el notebook genera su propio **dataset sintetico** de formas geometricas con mascaras anotadas. Esto permite controlar exactamente la dificultad, el desbalance de clases y las condiciones de entrenamiento.

---

## Contenidos

1. Fundamentos: tipos de segmentacion y formulacion matematica
2. Dataset sintetico con mascaras pixel-a-pixel
3. Arquitectura FCN basica (Fully Convolutional Network)
4. Arquitectura U-Net con skip connections
5. Funciones de perdida para segmentacion
6. Metricas de evaluacion
7. Data augmentation sincronizada imagen + mascara
8. Experimento: impacto de la funcion de perdida
9. Comparativa de arquitecturas

---

## 1. Fundamentos: tipos de segmentacion y formulacion matematica

### Definicion del problema

Dada una imagen $I \in \mathbb{R}^{H \times W \times C}$, la segmentacion semantica asigna una etiqueta de clase $y_{ij} \in \{0, 1, \ldots, K-1\}$ a cada pixel $(i, j)$:

$$f_\theta : \mathbb{R}^{H \times W \times C} \rightarrow \{0, \ldots, K-1\}^{H \times W}$$

En la practica, la red produce un **mapa de probabilidades** de forma $(H, W, K)$, donde cada canal $k$ contiene la probabilidad de que cada pixel pertenezca a la clase $k$. La etiqueta final se obtiene con:

$$\hat{y}_{ij} = \arg\max_k \; p_{ijk}$$

### Tipos de segmentacion

| Tipo | Salida | Distingue instancias | Ejemplo |
|------|--------|---------------------|----------|
| **Semantica** | Mascara de clase por pixel | No | Todo el cielo = clase 0 |
| **Instancias** | Mascara por objeto individual | Si | Persona 1, Persona 2... |
| **Panoptica** | Semantica + instancias unificadas | Si | Mask R-CNN, OneFormer |

Este notebook se centra en la **segmentacion semantica**, que es la base de todas las variantes.

### La diferencia clave con la clasificacion

En clasificacion, la red colapsa la dimension espacial (con pooling global) para producir un vector. En segmentacion, la dimension espacial debe **preservarse**: la salida tiene la misma resolucion que la entrada. Esto requiere arquitecturas encoder-decoder que compriman y luego expandan la representacion.

---

## 2. Dataset sintetico con mascaras pixel-a-pixel

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image, ImageDraw
import random

# Reproducibilidad
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo : {device}')
print(f'PyTorch     : {torch.__version__}')

In [ ]:
# Clases del dataset sintetico
# 0: Fondo (background)  -- clase mayoritaria, genera desbalance
# 1: Circulo             -- objetos medianos
# 2: Rectangulo          -- objetos medianos
# 3: Triangulo           -- objetos pequeños, mas dificil

N_CLASSES  = 4
IMG_H      = 128
IMG_W      = 128

CLASS_COLORS = [
    (30,  30,  30),   # 0: fondo (gris oscuro)
    (220, 60,  60),   # 1: circulo (rojo)
    (60,  120, 220),  # 2: rectangulo (azul)
    (60,  200, 80),   # 3: triangulo (verde)
]

CLASS_NAMES = ['Fondo', 'Circulo', 'Rectangulo', 'Triangulo']


def generate_sample(img_h=128, img_w=128, n_shapes=3, seed=None):
    """
    Genera un par (imagen RGB, mascara de segmentacion) de forma sintetica.

    Proceso:
    - Se dibuja un fondo con textura de ruido suave.
    - Se superponen n_shapes formas aleatorias (circulos, rectangulos, triangulos).
    - La mascara registra la clase de cada pixel segun la forma mas reciente
      (las formas se sobreescriben entre si, simulando oclusion parcial).

    Retorna:
    - img : np.array (H, W, 3) uint8, imagen RGB
    - mask: np.array (H, W)    int64, mascara con etiquetas de clase por pixel
    """
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    # Imagen con textura de fondo (ruido suave)
    bg_color = np.random.randint(20, 60)
    img_arr  = np.ones((img_h, img_w, 3), dtype=np.uint8) * bg_color
    img_arr  += np.random.randint(0, 15, img_arr.shape, dtype=np.uint8)
    img_pil  = Image.fromarray(img_arr)
    draw     = ImageDraw.Draw(img_pil)

    mask_arr = np.zeros((img_h, img_w), dtype=np.int64)  # clase 0 = fondo

    for _ in range(n_shapes):
        shape_type = random.choice([1, 2, 3])  # clase de la forma
        color      = CLASS_COLORS[shape_type]
        # Variar ligeramente el color para que no sea perfectamente uniforme
        color_var  = tuple(max(0, min(255, c + random.randint(-20, 20))) for c in color)

        size = random.randint(15, 45)
        x0   = random.randint(5, img_w - size - 5)
        y0   = random.randint(5, img_h - size - 5)
        x1   = x0 + size
        y1   = y0 + size

        if shape_type == 1:  # Circulo
            draw.ellipse([x0, y0, x1, y1], fill=color_var)
            # Actualizar mascara: todos los pixeles dentro del bounding box del circulo
            for py in range(y0, min(y1 + 1, img_h)):
                for px in range(x0, min(x1 + 1, img_w)):
                    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
                    rx, ry = (x1 - x0) / 2, (y1 - y0) / 2
                    if rx > 0 and ry > 0 and ((px - cx)/rx)**2 + ((py - cy)/ry)**2 <= 1:
                        mask_arr[py, px] = 1

        elif shape_type == 2:  # Rectangulo
            draw.rectangle([x0, y0, x1, y1], fill=color_var)
            mask_arr[y0:y1+1, x0:x1+1] = 2

        elif shape_type == 3:  # Triangulo
            cx   = (x0 + x1) // 2
            pts  = [(cx, y0), (x0, y1), (x1, y1)]
            draw.polygon(pts, fill=color_var)
            # Rasterizacion del triangulo en la mascara
            for py in range(y0, min(y1 + 1, img_h)):
                t    = (py - y0) / max(y1 - y0, 1)
                xleft  = int(cx + t * (x0 - cx))
                xright = int(cx + t * (x1 - cx))
                for px in range(max(0, xleft), min(img_w, xright + 1)):
                    mask_arr[py, px] = 3

    img = np.array(img_pil, dtype=np.uint8)
    return img, mask_arr


# Verificar generacion
img_test, mask_test = generate_sample(seed=42)
print(f'Imagen: {img_test.shape}, dtype: {img_test.dtype}')
print(f'Mascara: {mask_test.shape}, dtype: {mask_test.dtype}')
print(f'Clases presentes: {np.unique(mask_test)}')
for c in range(N_CLASSES):
    pct = (mask_test == c).mean() * 100
    print(f'  Clase {c} ({CLASS_NAMES[c]}): {pct:.1f}% de los pixeles')

In [ ]:
def visualize_sample(img, mask, title=''):
    """
    Muestra la imagen original, la mascara codificada por color y una superposicion.
    La superposicion permite verificar que la mascara se alinea correctamente con la imagen.
    """
    mask_rgb = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for c, color in enumerate(CLASS_COLORS):
        mask_rgb[mask == c] = color

    overlay = (img.astype(float) * 0.6 + mask_rgb.astype(float) * 0.4).astype(np.uint8)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(img);            axes[0].set_title('Imagen RGB');              axes[0].axis('off')
    axes[1].imshow(mask_rgb);       axes[1].set_title('Mascara (color por clase)'); axes[1].axis('off')
    axes[2].imshow(overlay);        axes[2].set_title('Superposicion');            axes[2].axis('off')

    # Leyenda de clases
    handles = [mpatches.Patch(color=[c/255 for c in col], label=name)
               for col, name in zip(CLASS_COLORS, CLASS_NAMES)]
    axes[1].legend(handles=handles, loc='lower right', fontsize=8)

    if title:
        plt.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()


visualize_sample(img_test, mask_test, 'Ejemplo del dataset sintetico')

In [ ]:
class ShapeSegDataset(Dataset):
    """
    Dataset de segmentacion semantica con formas geometricas sinteticas.

    Genera los pares (imagen, mascara) de forma procedural en __getitem__,
    lo que garantiza variabilidad infinita sin necesidad de almacenamiento en disco.
    El seed por indice garantiza reproducibilidad: el mismo indice siempre
    produce el mismo par imagen-mascara.
    """
    def __init__(self, n_samples=1000, img_h=128, img_w=128,
                 n_shapes=3, transform=None):
        self.n_samples  = n_samples
        self.img_h      = img_h
        self.img_w      = img_w
        self.n_shapes   = n_shapes
        self.transform  = transform

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        img, mask = generate_sample(self.img_h, self.img_w, self.n_shapes, seed=idx)

        # Aplicar transformaciones opcionales (imagen Y mascara de forma sincronizada)
        if self.transform is not None:
            img, mask = self.transform(img, mask)
        else:
            # Conversion por defecto: numpy -> tensor
            img  = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
            mask = torch.from_numpy(mask).long()

        return img, mask


# Crear datasets
train_dataset = ShapeSegDataset(n_samples=1200, img_h=IMG_H, img_w=IMG_W, n_shapes=3)
val_dataset   = ShapeSegDataset(n_samples=200,  img_h=IMG_H, img_w=IMG_W, n_shapes=3)
test_dataset  = ShapeSegDataset(n_samples=200,  img_h=IMG_H, img_w=IMG_W, n_shapes=3)

# Los indices del test se desplazan para que no coincidan con train/val
# (se consigue usando semillas distintas en generate_sample)
class OffsetDataset(Dataset):
    def __init__(self, base_dataset, offset):
        self.base   = base_dataset
        self.offset = offset
    def __len__(self):      return len(self.base)
    def __getitem__(self, idx):
        img, mask = generate_sample(IMG_H, IMG_W, 3, seed=idx + self.offset)
        img  = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        mask = torch.from_numpy(mask).long()
        return img, mask

val_dataset  = OffsetDataset(val_dataset,  offset=5000)
test_dataset = OffsetDataset(test_dataset, offset=10000)

BATCH_SIZE   = 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Ejemplos de entrenamiento : {len(train_dataset):,}')
print(f'Ejemplos de validacion    : {len(val_dataset):,}')
print(f'Ejemplos de prueba        : {len(test_dataset):,}')

imgs, masks = next(iter(train_loader))
print(f'\nForma del batch de imagenes : {imgs.shape}   (B, C, H, W)')
print(f'Forma del batch de mascaras : {masks.shape}  (B, H, W)')
print(f'Rango de pixeles de imagen  : [{imgs.min():.2f}, {imgs.max():.2f}]')
print(f'Clases presentes en batch   : {masks.unique().tolist()}')

In [ ]:
# Analisis del desbalance de clases
# Fundamental para elegir correctamente la funcion de perdida
class_counts = torch.zeros(N_CLASSES)
for _, masks_b in train_loader:
    for c in range(N_CLASSES):
        class_counts[c] += (masks_b == c).sum()

total_pixels = class_counts.sum()
class_freqs  = class_counts / total_pixels

print('Distribucion de pixeles por clase en el set de entrenamiento:')
for c in range(N_CLASSES):
    barra = '#' * int(class_freqs[c].item() * 60)
    print(f'  {CLASS_NAMES[c]:<12} : {class_freqs[c]*100:5.1f}%  {barra}')

print()
print('Observacion: el fondo domina la imagen.')
print('Esto es tipico en segmentacion real (celulas, lesiones, objetos sobre fondo).')
print('Una perdida naive (Cross-Entropy sin pesos) sesga el modelo hacia predecir fondo.')

# Calcular pesos de clase inversos a la frecuencia
class_weights = 1.0 / (class_freqs + 1e-6)
class_weights = class_weights / class_weights.sum() * N_CLASSES  # normalizar
class_weights = class_weights.to(device)
print(f'\nPesos de clase (inverse frequency weighting):')
for c in range(N_CLASSES):
    print(f'  {CLASS_NAMES[c]:<12} : {class_weights[c].item():.3f}')

In [ ]:
# Visualizar varios ejemplos del dataset
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for i in range(5):
    img_np  = (imgs[i].permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    mask_np = masks[i].numpy()

    mask_rgb = np.zeros((*mask_np.shape, 3), dtype=np.uint8)
    for c, col in enumerate(CLASS_COLORS):
        mask_rgb[mask_np == c] = col

    axes[0, i].imshow(img_np);   axes[0, i].axis('off')
    axes[1, i].imshow(mask_rgb); axes[1, i].axis('off')

axes[0, 0].set_ylabel('Imagen', fontsize=11, rotation=0, labelpad=40, va='center')
axes[1, 0].set_ylabel('Mascara', fontsize=11, rotation=0, labelpad=40, va='center')
plt.suptitle('Muestras del dataset sintetico de segmentacion', fontsize=13)
plt.tight_layout()
plt.show()

---

## 3. Arquitectura FCN basica (Fully Convolutional Network)

Long et al. (2015) propusieron la primera arquitectura totalmente convolucional para segmentacion densa. La idea clave: sustituir las capas densas finales de una CNN de clasificacion por convoluciones $1\times1$, y recuperar la resolucion espacial con **upsampling bilineal**.

```
Imagen (3, H, W)
  -> Conv + Pool  (encoder, reduce resolucion)
  -> Conv 1x1     (clasificador por canal)
  -> Upsample     (restaura resolucion original)
  -> Softmax      (probabilidades por clase por pixel)
```

**Limitacion principal:** el upsampling directo desde una resolucion muy reducida produce mascaras con bordes difusos y poco detalle. Esta limitacion motivo el desarrollo de U-Net.

In [ ]:
class ConvBnRelu(nn.Module):
    """Bloque basico: Conv2d -> BatchNorm -> ReLU. Reutilizado en todas las arquitecturas."""
    def __init__(self, in_ch, out_ch, kernel=3, padding=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=kernel, padding=padding, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class SimpleFCN(nn.Module):
    """
    FCN simplificada para segmentacion semantica.

    Encoder: extrae caracteristicas reduciendo la resolucion espacial con MaxPool.
    Clasificador: Conv 1x1 que produce K mapas (uno por clase) sin cambiar resolucion.
    Decoder: upsampling bilineal para restaurar la resolucion original.

    El upsampling bilineal NO tiene parametros aprendibles: es una interpolacion fija.
    Esto es la limitacion principal de la FCN pura.
    """
    def __init__(self, n_classes=4, in_ch=3):
        super().__init__()

        # Encoder: 4 bloques con reduccion de resolucion x2 cada uno
        # (H, W) -> (H/2, W/2) -> (H/4, W/4) -> (H/8, W/8) -> (H/16, W/16)
        self.enc1 = nn.Sequential(ConvBnRelu(in_ch, 32),  ConvBnRelu(32, 32),  nn.MaxPool2d(2))
        self.enc2 = nn.Sequential(ConvBnRelu(32, 64),     ConvBnRelu(64, 64),  nn.MaxPool2d(2))
        self.enc3 = nn.Sequential(ConvBnRelu(64, 128),    ConvBnRelu(128, 128), nn.MaxPool2d(2))
        self.enc4 = nn.Sequential(ConvBnRelu(128, 256),   ConvBnRelu(256, 256), nn.MaxPool2d(2))

        # Clasificador: Conv 1x1 produce K canales (probabilidades por clase)
        # Equivalente a una capa densa aplicada independientemente a cada pixel
        self.classifier = nn.Conv2d(256, n_classes, kernel_size=1)

    def forward(self, x):
        h, w = x.shape[2], x.shape[3]  # guardar resolucion original

        x = self.enc1(x)   # (B, 32,  H/2,  W/2)
        x = self.enc2(x)   # (B, 64,  H/4,  W/4)
        x = self.enc3(x)   # (B, 128, H/8,  W/8)
        x = self.enc4(x)   # (B, 256, H/16, W/16)

        x = self.classifier(x)   # (B, n_classes, H/16, W/16)

        # Upsample bilineal: restaura la resolucion de la imagen original
        # align_corners=False es el comportamiento por defecto y mas correcto
        x = F.interpolate(x, size=(h, w), mode='bilinear', align_corners=False)

        return x  # (B, n_classes, H, W)  - logits, sin softmax


fcn = SimpleFCN(n_classes=N_CLASSES).to(device)

n_params = sum(p.numel() for p in fcn.parameters() if p.requires_grad)
print(f'Parametros FCN: {n_params:,}')

# Verificar forma de salida
with torch.no_grad():
    dummy    = torch.randn(2, 3, IMG_H, IMG_W).to(device)
    out_fcn  = fcn(dummy)
    print(f'Entrada : {dummy.shape}')
    print(f'Salida  : {out_fcn.shape}   (B, n_classes, H, W) - logits')

---

## 4. Arquitectura U-Net con skip connections

Ronneberger et al. (2015) propusieron U-Net para segmentacion de imagenes biomedicas. Resuelve el problema de detalle de la FCN mediante **conexiones de salto (skip connections)**: en cada nivel, la salida del encoder se concatena directamente con la entrada del decoder correspondiente.

```
Encoder (contraccion):          Decoder (expansion):
  x    -> [E1] -> s1               s4 --------> [D4] -> up4
  s1   -> [E2] -> s2               up4 + s3 -> [D3] -> up3
  s2   -> [E3] -> s3               up3 + s2 -> [D2] -> up2
  s3   -> [E4] -> bottleneck       up2 + s1 -> [D1] -> salida
```

**Por que funcionan las skip connections:**
- El bottleneck captura **contexto semantico global** (que hay en la imagen).
- Las skip connections aportan **detalle espacial fino** (donde estan exactamente los bordes).
- Sin ellas, el decoder debe reconstruir los detalles solo desde el bottleneck comprimido.

In [ ]:
class DoubleConv(nn.Module):
    """Bloque doble de convolucion tipico de U-Net: Conv -> BN -> ReLU -> Conv -> BN -> ReLU."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    """
    U-Net para segmentacion semantica.

    Arquitectura simetrica encoder-decoder con skip connections.
    El decoder combina la informacion semantica (del nivel superior)
    con la informacion espacial de alta resolucion (skip del encoder).

    Parametros:
    - in_ch    : canales de la imagen de entrada (3 para RGB)
    - n_classes: numero de clases de segmentacion
    - features : numero base de filtros (se duplica en cada nivel)
    """
    def __init__(self, in_ch=3, n_classes=4, features=32):
        super().__init__()

        # ---- Encoder ----
        # Cada bloque extrae caracteristicas y luego MaxPool reduce la resolucion
        self.enc1 = DoubleConv(in_ch,         features)      # (B, 32,  H,   W)
        self.enc2 = DoubleConv(features,      features*2)    # (B, 64,  H/2, W/2)
        self.enc3 = DoubleConv(features*2,    features*4)    # (B, 128, H/4, W/4)
        self.enc4 = DoubleConv(features*4,    features*8)    # (B, 256, H/8, W/8)
        self.pool = nn.MaxPool2d(2)

        # ---- Bottleneck ----
        # El nivel mas comprimido: captura el contexto semantico global
        self.bottleneck = DoubleConv(features*8, features*16)  # (B, 512, H/16, W/16)

        # ---- Decoder ----
        # ConvTranspose2d (upsampling aprendible) + DoubleConv sobre la concatenacion
        # La entrada a cada DoubleConv es features*k (skip) + features*k (upsampled) = features*2k
        self.up4    = nn.ConvTranspose2d(features*16, features*8,  kernel_size=2, stride=2)
        self.dec4   = DoubleConv(features*16, features*8)   # 256+256 -> 256

        self.up3    = nn.ConvTranspose2d(features*8,  features*4,  kernel_size=2, stride=2)
        self.dec3   = DoubleConv(features*8,  features*4)   # 128+128 -> 128

        self.up2    = nn.ConvTranspose2d(features*4,  features*2,  kernel_size=2, stride=2)
        self.dec2   = DoubleConv(features*4,  features*2)   # 64+64   -> 64

        self.up1    = nn.ConvTranspose2d(features*2,  features,    kernel_size=2, stride=2)
        self.dec1   = DoubleConv(features*2,  features)     # 32+32   -> 32

        # ---- Cabeza de clasificacion ----
        # Conv 1x1: mapea los features finales a K logits por pixel
        self.out_conv = nn.Conv2d(features, n_classes, kernel_size=1)

    def forward(self, x):
        # Encoder: guardar los mapas de cada nivel para las skip connections
        s1 = self.enc1(x)           # (B, 32,  H,    W)
        s2 = self.enc2(self.pool(s1))  # (B, 64,  H/2,  W/2)
        s3 = self.enc3(self.pool(s2))  # (B, 128, H/4,  W/4)
        s4 = self.enc4(self.pool(s3))  # (B, 256, H/8,  W/8)

        # Bottleneck
        bn = self.bottleneck(self.pool(s4))  # (B, 512, H/16, W/16)

        # Decoder: upsampling + concatenacion con skip + doble conv
        x = self.dec4(torch.cat([self.up4(bn), s4], dim=1))  # (B, 256, H/8,  W/8)
        x = self.dec3(torch.cat([self.up3(x),  s3], dim=1))  # (B, 128, H/4,  W/4)
        x = self.dec2(torch.cat([self.up2(x),  s2], dim=1))  # (B, 64,  H/2,  W/2)
        x = self.dec1(torch.cat([self.up1(x),  s1], dim=1))  # (B, 32,  H,    W)

        return self.out_conv(x)   # (B, n_classes, H, W) - logits


unet = UNet(in_ch=3, n_classes=N_CLASSES, features=32).to(device)

n_params = sum(p.numel() for p in unet.parameters() if p.requires_grad)
print(f'Parametros U-Net : {n_params:,}')

with torch.no_grad():
    out_unet = unet(dummy)
    print(f'Entrada : {dummy.shape}')
    print(f'Salida  : {out_unet.shape}   (B, n_classes, H, W) - logits')

---

## 5. Funciones de perdida para segmentacion

La eleccion de la funcion de perdida es critica en segmentacion, especialmente cuando hay **desbalance de clases** (el fondo ocupa la mayoria de los pixeles).

In [ ]:
class SegmentationLosses:
    """
    Coleccion de funciones de perdida para segmentacion semantica.
    Todas reciben logits (sin softmax) y mascaras de etiquetas enteras.
    """

    @staticmethod
    def cross_entropy(logits, targets, weights=None):
        """
        Cross-Entropy pixelwise (estandar).

        Cada pixel contribuye igual a la perdida. Con desbalance severo,
        el modelo aprende a predecir siempre la clase mayoritaria.

        Con class weights, las clases minoritarias tienen mayor penalizacion
        cuando se clasifican incorrectamente.
        """
        return F.cross_entropy(logits, targets, weight=weights)

    @staticmethod
    def dice_loss(logits, targets, n_classes=4, smooth=1.0):
        """
        Dice Loss (equivalente a 1 - F1 score sobre regiones).

        Formula: Dice = 2|A cap B| / (|A| + |B|)
        Dice Loss = 1 - Dice

        Ventajas:
        - Invariante al tamano de la region: trata igual una clase con
          pocos pixeles que una con muchos.
        - Optimiza directamente la superposicion entre prediccion y GT.
        - Excelente para objetos pequeños y datasets desbalanceados.

        smooth evita division por cero cuando la clase no esta presente.
        """
        probs    = F.softmax(logits, dim=1)              # (B, K, H, W)
        # One-hot encoding de la mascara: (B, H, W) -> (B, K, H, W)
        targets_oh = F.one_hot(targets, n_classes).permute(0, 3, 1, 2).float()

        dice_per_class = []
        for c in range(n_classes):
            p = probs[:, c]        # (B, H, W) probabilidades de clase c
            t = targets_oh[:, c]  # (B, H, W) GT binario de clase c

            intersection = (p * t).sum(dim=(1, 2))
            union        = p.sum(dim=(1, 2)) + t.sum(dim=(1, 2))
            dice_c       = (2 * intersection + smooth) / (union + smooth)
            dice_per_class.append(dice_c.mean())

        # Promedio sobre clases (macro-averaging)
        return 1 - torch.stack(dice_per_class).mean()

    @staticmethod
    def iou_loss(logits, targets, n_classes=4, smooth=1.0):
        """
        IoU Loss (Jaccard Loss): optimiza directamente la metrica mIoU.

        Formula: IoU = |A cap B| / |A cup B| = |A cap B| / (|A| + |B| - |A cap B|)
        IoU Loss = 1 - IoU

        Mas estricta que Dice: penaliza mas los falsos positivos y negativos.
        """
        probs      = F.softmax(logits, dim=1)
        targets_oh = F.one_hot(targets, n_classes).permute(0, 3, 1, 2).float()

        iou_per_class = []
        for c in range(n_classes):
            p            = probs[:, c]
            t            = targets_oh[:, c]
            intersection = (p * t).sum(dim=(1, 2))
            union        = p.sum(dim=(1, 2)) + t.sum(dim=(1, 2)) - intersection
            iou_c        = (intersection + smooth) / (union + smooth)
            iou_per_class.append(iou_c.mean())

        return 1 - torch.stack(iou_per_class).mean()

    @staticmethod
    def focal_loss(logits, targets, gamma=2.0, weights=None):
        """
        Focal Loss (Lin et al., 2017).

        Modifica la CE multiplicando por (1 - p)^gamma:
        - Pixeles bien clasificados (p alta): (1-p)^gamma ~= 0, contribuyen poco.
        - Pixeles dificiles (p baja): (1-p)^gamma ~= 1, contribuyen normalmente.

        Efecto: enfoca el entrenamiento en los pixeles dificiles (bordes, clases raras).
        gamma=0 recupera la CE estandar.
        """
        ce   = F.cross_entropy(logits, targets, weight=weights, reduction='none')
        probs = F.softmax(logits, dim=1)
        # Probabilidad de la clase correcta para cada pixel
        pt   = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        focal_weight = (1 - pt) ** gamma
        return (focal_weight * ce).mean()

    @staticmethod
    def combined_loss(logits, targets, n_classes=4, weights=None,
                      lambda_dice=0.5):
        """
        Perdida combinada: CE + lambda * Dice.

        Practica habitual en segmentacion:
        - CE aporta estabilidad de gradiente y convergencia rapida.
        - Dice optimiza directamente la superposicion, robusta al desbalance.
        - La combinacion es mas robusta que cualquiera de las dos solas.
        """
        ce   = F.cross_entropy(logits, targets, weight=weights)
        dice = SegmentationLosses.dice_loss(logits, targets, n_classes)
        return ce + lambda_dice * dice


# Demostrar las perdidas sobre un batch sintetico
losses_obj = SegmentationLosses()

with torch.no_grad():
    batch_imgs, batch_masks = next(iter(train_loader))
    batch_imgs, batch_masks = batch_imgs.to(device), batch_masks.to(device)
    logits = unet(batch_imgs)

print('Perdidas sobre el primer batch (modelo sin entrenar):')
print(f'  Cross-Entropy          : {SegmentationLosses.cross_entropy(logits, batch_masks):.4f}')
print(f'  Weighted CE            : {SegmentationLosses.cross_entropy(logits, batch_masks, class_weights):.4f}')
print(f'  Dice Loss              : {SegmentationLosses.dice_loss(logits, batch_masks, N_CLASSES):.4f}')
print(f'  IoU Loss               : {SegmentationLosses.iou_loss(logits, batch_masks, N_CLASSES):.4f}')
print(f'  Focal Loss (gamma=2)   : {SegmentationLosses.focal_loss(logits, batch_masks):.4f}')
print(f'  Combined (CE + Dice)   : {SegmentationLosses.combined_loss(logits, batch_masks, N_CLASSES):.4f}')

---

## 6. Metricas de evaluacion

In [ ]:
class SegmentationMetrics:
    """
    Metricas estandar para segmentacion semantica.
    Todas operan sobre predicciones y mascaras de clase entera.
    """

    @staticmethod
    def pixel_accuracy(preds, targets):
        """
        Accuracy a nivel de pixel: fraccion de pixeles correctamente clasificados.

        Limitacion: en datasets desbalanceados un modelo que predice siempre
        la clase mayoritaria (fondo) puede tener accuracy alta.
        """
        correct = (preds == targets).sum().item()
        total   = targets.numel()
        return correct / total

    @staticmethod
    def iou_per_class(preds, targets, n_classes):
        """
        Intersection over Union por clase.

        IoU_c = TP_c / (TP_c + FP_c + FN_c)
              = |prediccion_c cap GT_c| / |prediccion_c cup GT_c|

        Retorna NaN para clases no presentes en el batch (para no sesgar el promedio).
        """
        ious = []
        for c in range(n_classes):
            pred_c = (preds   == c)
            gt_c   = (targets == c)
            intersection = (pred_c & gt_c).sum().item()
            union        = (pred_c | gt_c).sum().item()
            if union == 0:
                ious.append(float('nan'))  # clase no presente
            else:
                ious.append(intersection / union)
        return ious

    @staticmethod
    def mean_iou(preds, targets, n_classes):
        """
        mIoU: promedio de IoU sobre las clases presentes.
        Es la metrica estandar en benchmarks de segmentacion (PASCAL VOC, COCO, Cityscapes).
        """
        ious  = SegmentationMetrics.iou_per_class(preds, targets, n_classes)
        valid = [v for v in ious if not np.isnan(v)]
        return np.mean(valid) if valid else 0.0

    @staticmethod
    def dice_per_class(preds, targets, n_classes, smooth=1e-6):
        """
        Dice coefficient por clase (equivalente a F1-score sobre regiones).

        Dice_c = 2*TP_c / (2*TP_c + FP_c + FN_c)
        """
        dices = []
        for c in range(n_classes):
            pred_c = (preds   == c).float()
            gt_c   = (targets == c).float()
            intersection = (pred_c * gt_c).sum().item()
            denom        = pred_c.sum().item() + gt_c.sum().item()
            if denom == 0:
                dices.append(float('nan'))
            else:
                dices.append((2 * intersection + smooth) / (denom + smooth))
        return dices

    @staticmethod
    def mean_dice(preds, targets, n_classes):
        dices = SegmentationMetrics.dice_per_class(preds, targets, n_classes)
        valid = [v for v in dices if not np.isnan(v)]
        return np.mean(valid) if valid else 0.0

    @staticmethod
    def confusion_matrix(preds, targets, n_classes):
        """
        Matriz de confusion de segmentacion: C[i,j] = pixeles de clase real i predichos como j.
        """
        cm = np.zeros((n_classes, n_classes), dtype=np.int64)
        for t, p in zip(targets.flatten().tolist(), preds.flatten().tolist()):
            cm[int(t), int(p)] += 1
        return cm


# Demostrar metricas
with torch.no_grad():
    logits = unet(batch_imgs)
    preds  = logits.argmax(dim=1).cpu()
    gt     = batch_masks.cpu()

metrics = SegmentationMetrics()
pa      = metrics.pixel_accuracy(preds, gt)
miou    = metrics.mean_iou(preds, gt, N_CLASSES)
mdice   = metrics.mean_dice(preds, gt, N_CLASSES)
ious    = metrics.iou_per_class(preds, gt, N_CLASSES)

print('Metricas sobre el primer batch (modelo sin entrenar):')
print(f'  Pixel Accuracy : {pa:.4f}')
print(f'  mIoU           : {miou:.4f}')
print(f'  Mean Dice      : {mdice:.4f}')
print()
print('  IoU por clase:')
for c, iou in enumerate(ious):
    print(f'    {CLASS_NAMES[c]:<12}: {iou:.4f}' if not np.isnan(iou) else f'    {CLASS_NAMES[c]:<12}: N/A')

In [ ]:
def evaluate_model(model, loader, n_classes, device):
    """
    Evalua un modelo de segmentacion sobre un DataLoader completo.
    Acumula predicciones y calcula metricas globales (no por batch).
    """
    model.eval()
    all_preds  = []
    all_targets = []

    with torch.no_grad():
        for imgs, masks in loader:
            imgs  = imgs.to(device)
            logits = model(imgs)
            preds  = logits.argmax(dim=1).cpu()
            all_preds.append(preds)
            all_targets.append(masks)

    all_preds   = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)

    metrics = SegmentationMetrics()
    return {
        'pixel_acc' : metrics.pixel_accuracy(all_preds, all_targets),
        'mIoU'      : metrics.mean_iou(all_preds, all_targets, n_classes),
        'mean_dice' : metrics.mean_dice(all_preds, all_targets, n_classes),
        'iou_per_class': metrics.iou_per_class(all_preds, all_targets, n_classes),
        'preds'     : all_preds,
        'targets'   : all_targets,
    }

---

## 7. Data augmentation sincronizada imagen + mascara

La augmentacion en segmentacion tiene una restriccion fundamental: **toda transformacion geometrica debe aplicarse identicamente a la imagen y a su mascara**. Si la imagen se voltea, la mascara debe voltearse exactamente igual. Las transformaciones de color o ruido se aplican solo a la imagen.

In [ ]:
class SyncAugmentation:
    """
    Augmentacion sincronizada para segmentacion semantica.

    Transformaciones GEOMETRICAS (se aplican identicamente a imagen y mascara):
    - Flip horizontal / vertical
    - Rotacion aleatoria
    - Crop aleatorio + resize

    Transformaciones de COLOR (solo a la imagen, la mascara no cambia):
    - Jitter de brillo, contraste, saturacion
    - Ruido gaussiano
    """

    def __init__(self,
                 hflip_prob   = 0.5,
                 vflip_prob   = 0.2,
                 rotate_prob  = 0.5,
                 rotate_range = 15,
                 crop_prob    = 0.5,
                 crop_scale   = (0.7, 1.0),
                 noise_std    = 0.02,
                 img_h        = 128,
                 img_w        = 128):
        self.hflip_prob   = hflip_prob
        self.vflip_prob   = vflip_prob
        self.rotate_prob  = rotate_prob
        self.rotate_range = rotate_range
        self.crop_prob    = crop_prob
        self.crop_scale   = crop_scale
        self.noise_std    = noise_std
        self.img_h        = img_h
        self.img_w        = img_w

    def __call__(self, img_np, mask_np):
        """
        Aplica augmentacion sincronizada.

        Entrada:
        - img_np  : np.array (H, W, 3) uint8
        - mask_np : np.array (H, W)    int64

        Salida:
        - img_tensor  : torch.Tensor (3, H, W) float32 en [0, 1]
        - mask_tensor : torch.Tensor (H, W)    int64
        """
        img  = Image.fromarray(img_np)
        # La mascara se trata como imagen de modo 'L' (un solo canal, entero)
        mask = Image.fromarray(mask_np.astype(np.uint8), mode='L')

        # 1. Flip horizontal
        if random.random() < self.hflip_prob:
            img  = img.transpose(Image.FLIP_LEFT_RIGHT)
            mask = mask.transpose(Image.FLIP_LEFT_RIGHT)

        # 2. Flip vertical
        if random.random() < self.vflip_prob:
            img  = img.transpose(Image.FLIP_TOP_BOTTOM)
            mask = mask.transpose(Image.FLIP_TOP_BOTTOM)

        # 3. Rotacion aleatoria (mismo angulo para imagen y mascara)
        if random.random() < self.rotate_prob:
            angle = random.uniform(-self.rotate_range, self.rotate_range)
            img   = img.rotate(angle,  resample=Image.BILINEAR, expand=False)
            mask  = mask.rotate(angle, resample=Image.NEAREST,  expand=False)
            # La mascara usa NEAREST para no interpolar etiquetas de clase

        # 4. Crop aleatorio + resize (mismo crop para imagen y mascara)
        if random.random() < self.crop_prob:
            scale   = random.uniform(*self.crop_scale)
            new_h   = int(self.img_h * scale)
            new_w   = int(self.img_w * scale)
            top     = random.randint(0, self.img_h - new_h)
            left    = random.randint(0, self.img_w - new_w)
            img     = img.crop((left, top, left + new_w, top + new_h))
            mask    = mask.crop((left, top, left + new_w, top + new_h))
            # Resize de vuelta a la resolucion original
            img     = img.resize((self.img_w, self.img_h), Image.BILINEAR)
            mask    = mask.resize((self.img_w, self.img_h), Image.NEAREST)

        # 5. Ruido gaussiano solo en la imagen
        img_t = torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0
        if self.noise_std > 0:
            img_t = (img_t + torch.randn_like(img_t) * self.noise_std).clamp(0, 1)

        mask_t = torch.from_numpy(np.array(mask)).long()

        return img_t, mask_t


# Visualizar el efecto de la augmentacion
augment     = SyncAugmentation(img_h=IMG_H, img_w=IMG_W)
img_raw, mask_raw = generate_sample(seed=7)

fig, axes = plt.subplots(2, 5, figsize=(18, 7))

# Columna 0: original
mask_rgb0 = np.zeros((*mask_raw.shape, 3), dtype=np.uint8)
for c, col in enumerate(CLASS_COLORS): mask_rgb0[mask_raw == c] = col
axes[0, 0].imshow(img_raw);   axes[0, 0].set_title('Original', fontsize=10); axes[0, 0].axis('off')
axes[1, 0].imshow(mask_rgb0); axes[1, 0].axis('off')

# Columnas 1-4: versiones aumentadas
for col in range(1, 5):
    random.seed(col * 13)
    img_t, mask_t = augment(img_raw, mask_raw)
    img_aug  = (img_t.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    mask_aug = mask_t.numpy()

    mask_rgb_aug = np.zeros((*mask_aug.shape, 3), dtype=np.uint8)
    for c, color in enumerate(CLASS_COLORS): mask_rgb_aug[mask_aug == c] = color

    axes[0, col].imshow(img_aug);       axes[0, col].set_title(f'Aumentada {col}', fontsize=10); axes[0, col].axis('off')
    axes[1, col].imshow(mask_rgb_aug);  axes[1, col].axis('off')

axes[0, 0].set_ylabel('Imagen', fontsize=11, rotation=0, labelpad=40, va='center')
axes[1, 0].set_ylabel('Mascara', fontsize=11, rotation=0, labelpad=40, va='center')
plt.suptitle('Augmentacion sincronizada: la mascara sigue exactamente a la imagen', fontsize=13)
plt.tight_layout()
plt.show()

---

## 8. Entrenamiento: FCN vs. U-Net con perdida combinada

In [ ]:
def train_segmentation(model, train_loader, val_loader, epochs=30,
                       lr=1e-3, loss_fn='combined', n_classes=4,
                       class_weights=None, device='cpu'):
    """
    Bucle de entrenamiento para modelos de segmentacion semantica.

    Registra perdida de entrenamiento y mIoU de validacion por epoca.
    El scheduler ReduceLROnPlateau reduce el lr cuando mIoU no mejora,
    tecnica comun para escapar de minimos locales en la etapa final.
    """
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5, verbose=False
    )

    losses_obj = SegmentationLosses()
    history    = {'train_loss': [], 'val_miou': [], 'val_dice': []}

    best_miou  = 0.0

    for epoch in range(1, epochs + 1):
        # ---- Entrenamiento ----
        model.train()
        total_loss = 0.0

        for imgs, masks in train_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            logits = model(imgs)

            if loss_fn == 'ce':
                loss = losses_obj.cross_entropy(logits, masks)
            elif loss_fn == 'weighted_ce':
                loss = losses_obj.cross_entropy(logits, masks, class_weights)
            elif loss_fn == 'dice':
                loss = losses_obj.dice_loss(logits, masks, n_classes)
            elif loss_fn == 'focal':
                loss = losses_obj.focal_loss(logits, masks, gamma=2.0)
            elif loss_fn == 'combined':
                loss = losses_obj.combined_loss(logits, masks, n_classes, class_weights)
            else:
                raise ValueError(f'Loss desconocida: {loss_fn}')

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        # ---- Validacion ----
        val_metrics = evaluate_model(model, val_loader, n_classes, device)
        val_miou    = val_metrics['mIoU']
        val_dice    = val_metrics['mean_dice']

        scheduler.step(val_miou)

        history['train_loss'].append(avg_loss)
        history['val_miou'].append(val_miou)
        history['val_dice'].append(val_dice)

        if val_miou > best_miou:
            best_miou = val_miou

        if epoch % 10 == 0 or epoch == 1:
            lr_actual = optimizer.param_groups[0]['lr']
            print(f'Epoca {epoch:3d}/{epochs}  |  '
                  f'Loss: {avg_loss:.4f}  |  '
                  f'mIoU val: {val_miou:.4f}  |  '
                  f'Dice val: {val_dice:.4f}  |  '
                  f'lr: {lr_actual:.2e}')

    print(f'\nMejor mIoU de validacion: {best_miou:.4f}')
    return history


# Entrenar FCN con perdida combinada
fcn_model = SimpleFCN(n_classes=N_CLASSES).to(device)
print('Entrenando FCN con perdida combinada (CE + Dice)...')
history_fcn = train_segmentation(
    fcn_model, train_loader, val_loader,
    epochs=40, lr=1e-3, loss_fn='combined',
    n_classes=N_CLASSES, class_weights=class_weights, device=device
)

In [ ]:
# Entrenar U-Net con perdida combinada
unet_model = UNet(in_ch=3, n_classes=N_CLASSES, features=32).to(device)
print('Entrenando U-Net con perdida combinada (CE + Dice)...')
history_unet = train_segmentation(
    unet_model, train_loader, val_loader,
    epochs=40, lr=1e-3, loss_fn='combined',
    n_classes=N_CLASSES, class_weights=class_weights, device=device
)

In [ ]:
# Curvas de entrenamiento comparativas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_fcn['train_loss'],  label='FCN',   color='steelblue',  linewidth=2)
axes[0].plot(history_unet['train_loss'], label='U-Net', color='darkorange', linewidth=2)
axes[0].set_xlabel('Epoca'); axes[0].set_ylabel('Perdida')
axes[0].set_title('Perdida de entrenamiento')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history_fcn['val_miou'],  label='FCN',   color='steelblue',  linewidth=2)
axes[1].plot(history_unet['val_miou'], label='U-Net', color='darkorange', linewidth=2)
axes[1].set_xlabel('Epoca'); axes[1].set_ylabel('mIoU')
axes[1].set_title('mIoU en validacion')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('FCN vs. U-Net: curvas de entrenamiento', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
def visualize_predictions(model, dataset, indices, n_classes, device, title=''):
    """
    Muestra imagen / mascara GT / prediccion / superposicion para N ejemplos.
    Permite evaluar visualmente la calidad de la segmentacion:
    - Precision de bordes
    - Confusiones entre clases
    - Casos donde el modelo falla
    """
    model.eval()
    fig, axes = plt.subplots(len(indices), 4, figsize=(16, len(indices) * 4))
    if len(indices) == 1:
        axes = axes[np.newaxis, :]

    col_titles = ['Imagen', 'Mascara GT', 'Prediccion', 'Error (FP rojo, FN azul)']
    for ct, ax in zip(col_titles, axes[0]):
        ax.set_title(ct, fontsize=10)

    for row, idx in enumerate(indices):
        img_t, mask_t = dataset[idx]
        img_np        = (img_t.permute(1, 2, 0).numpy() * 255).astype(np.uint8)

        with torch.no_grad():
            logits = model(img_t.unsqueeze(0).to(device))
            pred   = logits.argmax(dim=1).squeeze(0).cpu()

        # Mascara GT codificada por color
        gt_rgb = np.zeros((*mask_t.shape, 3), dtype=np.uint8)
        for c, col in enumerate(CLASS_COLORS): gt_rgb[mask_t.numpy() == c] = col

        # Prediccion codificada por color
        pred_rgb = np.zeros((*pred.shape, 3), dtype=np.uint8)
        for c, col in enumerate(CLASS_COLORS): pred_rgb[pred.numpy() == c] = col

        # Mapa de error: FP en rojo, FN en azul, correcto en negro
        gt_np   = mask_t.numpy()
        pred_np = pred.numpy()
        error_map = np.zeros((*gt_np.shape, 3), dtype=np.uint8)
        # Falso positivo: predicho como objeto pero es fondo
        fp = (gt_np == 0) & (pred_np != 0)
        # Falso negativo: es objeto pero predicho como fondo
        fn = (gt_np != 0) & (pred_np == 0)
        # Error de clase: objeto predicho de clase equivocada
        wrong_class = (gt_np != 0) & (pred_np != 0) & (gt_np != pred_np)
        error_map[fp]          = [220, 60, 60]   # rojo
        error_map[fn]          = [60, 60, 220]   # azul
        error_map[wrong_class] = [220, 180, 60]  # amarillo

        axes[row, 0].imshow(img_np)
        axes[row, 1].imshow(gt_rgb)
        axes[row, 2].imshow(pred_rgb)
        axes[row, 3].imshow(error_map)
        for ax in axes[row]: ax.axis('off')

        # Calcular IoU del ejemplo
        miou_ex = SegmentationMetrics().mean_iou(pred, mask_t, n_classes)
        axes[row, 0].set_ylabel(f'mIoU: {miou_ex:.3f}', fontsize=9,
                                 rotation=0, labelpad=55, va='center')

    if title:
        plt.suptitle(title, fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()


visualize_predictions(unet_model, test_dataset, indices=[0, 1, 2, 3],
                      n_classes=N_CLASSES, device=device, title='U-Net: predicciones en test set')

In [ ]:
# Matriz de confusion sobre el test set completo
unet_model.eval()
all_preds_test, all_gt_test = [], []

with torch.no_grad():
    for imgs, masks in test_loader:
        logits = unet_model(imgs.to(device))
        preds  = logits.argmax(dim=1).cpu()
        all_preds_test.append(preds)
        all_gt_test.append(masks)

all_preds_test = torch.cat(all_preds_test)
all_gt_test    = torch.cat(all_gt_test)

cm = SegmentationMetrics().confusion_matrix(all_preds_test, all_gt_test, N_CLASSES)

# Normalizar por filas (clase real)
cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-6)

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right')
ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Clase predicha')
ax.set_ylabel('Clase real')
ax.set_title('Matriz de confusion normalizada - U-Net (test set)')

for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        color = 'white' if cm_norm[i, j] > 0.5 else 'black'
        ax.text(j, i, f'{cm_norm[i,j]:.2f}', ha='center', va='center',
                fontsize=11, color=color, fontweight='bold')

plt.tight_layout()
plt.show()

---

## 9. Comparativa de arquitecturas y funciones de perdida

In [ ]:
# Entrenar U-Net con distintas funciones de perdida para comparar
resultados = {}

for loss_name in ['ce', 'weighted_ce', 'dice', 'focal', 'combined']:
    print(f'\nEntrenando U-Net con perdida: {loss_name}')
    model_tmp = UNet(in_ch=3, n_classes=N_CLASSES, features=32).to(device)
    hist = train_segmentation(
        model_tmp, train_loader, val_loader,
        epochs=30, lr=1e-3, loss_fn=loss_name,
        n_classes=N_CLASSES, class_weights=class_weights, device=device
    )
    # Evaluar en test
    test_metrics = evaluate_model(model_tmp, test_loader, N_CLASSES, device)
    resultados[loss_name] = {
        'history'   : hist,
        'test_miou' : test_metrics['mIoU'],
        'test_dice' : test_metrics['mean_dice'],
        'test_pa'   : test_metrics['pixel_acc'],
        'iou_class' : test_metrics['iou_per_class'],
    }

In [ ]:
# Tabla de resultados en test set
print('Resultados en el test set - U-Net con distintas funciones de perdida')
print('=' * 80)
print(f'{"Perdida":<15} {"Pixel Acc":>10} {"mIoU":>10} {"Mean Dice":>10}  Clases IoU')
print('-' * 80)

for loss_name, res in resultados.items():
    iou_str = '  '.join([f'{v:.3f}' if not np.isnan(v) else ' N/A '
                          for v in res['iou_class']])
    print(f'{loss_name:<15} {res["test_pa"]:>10.4f} {res["test_miou"]:>10.4f} '
          f'{res["test_dice"]:>10.4f}  [{iou_str}]')

print()
print(f'Orden de clases IoU: {CLASS_NAMES}')

In [ ]:
# Curvas mIoU de validacion por funcion de perdida
colors = ['steelblue', 'darkorange', 'seagreen', 'mediumpurple', 'crimson']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for (loss_name, res), color in zip(resultados.items(), colors):
    axes[0].plot(res['history']['val_miou'],
                 label=loss_name, color=color, linewidth=2)
    axes[1].plot(res['history']['train_loss'],
                 label=loss_name, color=color, linewidth=2)

axes[0].set_title('mIoU de validacion por epoca')
axes[0].set_xlabel('Epoca'); axes[0].set_ylabel('mIoU')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

axes[1].set_title('Perdida de entrenamiento por epoca')
axes[1].set_xlabel('Epoca'); axes[1].set_ylabel('Perdida')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

plt.suptitle('Impacto de la funcion de perdida en U-Net', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Comparativa FCN vs U-Net en test
fcn_test  = evaluate_model(fcn_model,  test_loader, N_CLASSES, device)
unet_test = evaluate_model(unet_model, test_loader, N_CLASSES, device)

print('Comparativa final FCN vs. U-Net en test set (perdida combinada)')
print('=' * 60)
print(f'{"Metrica":<20} {"FCN":>12} {"U-Net":>12}')
print('-' * 60)
print(f'{"Pixel Accuracy":<20} {fcn_test["pixel_acc"]:>12.4f} {unet_test["pixel_acc"]:>12.4f}')
print(f'{"mIoU":<20} {fcn_test["mIoU"]:>12.4f} {unet_test["mIoU"]:>12.4f}')
print(f'{"Mean Dice":<20} {fcn_test["mean_dice"]:>12.4f} {unet_test["mean_dice"]:>12.4f}')
print()
print('IoU por clase:')
for c in range(N_CLASSES):
    fcn_iou  = fcn_test['iou_per_class'][c]
    unet_iou = unet_test['iou_per_class'][c]
    f_str    = f'{fcn_iou:.4f}'  if not np.isnan(fcn_iou)  else '  N/A '
    u_str    = f'{unet_iou:.4f}' if not np.isnan(unet_iou) else '  N/A '
    print(f'  {CLASS_NAMES[c]:<18} {f_str:>12} {u_str:>12}')

In [ ]:
# Grafica comparativa de IoU por clase
x       = np.arange(N_CLASSES)
width   = 0.35

fcn_ious  = [v if not np.isnan(v) else 0 for v in fcn_test['iou_per_class']]
unet_ious = [v if not np.isnan(v) else 0 for v in unet_test['iou_per_class']]

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, fcn_ious,  width, label='FCN',   color='steelblue',  alpha=0.85)
bars2 = ax.bar(x + width/2, unet_ious, width, label='U-Net', color='darkorange', alpha=0.85)

ax.bar_label(bars1, fmt='%.3f', padding=3, fontsize=9)
ax.bar_label(bars2, fmt='%.3f', padding=3, fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES)
ax.set_ylabel('IoU')
ax.set_ylim(0, 1.1)
ax.set_title('IoU por clase: FCN vs. U-Net (test set)')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Tabla resumen: cuando usar cada arquitectura y perdida
print('Guia de seleccion de arquitectura y funcion de perdida')
print('=' * 100)

arq_tabla = [
    ('FCN',          'Encoder CNN + Upsample bilineal', 'Bajo',   'Bajo',    'Prototipos rapidos, baseline'),
    ('U-Net',        'Encoder-Decoder + Skip Connections', 'Medio', 'Alto', 'Estandar en imagenes medicas y generales'),
    ('U-Net++',      'Skip Connections anidadas',       'Medio',  'Muy Alto','Tareas que requieren maxima precision de borde'),
    ('DeepLab v3+',  'Dilated Conv + ASPP + Decoder',  'Alto',   'Muy Alto','Escenas complejas, multiples escalas'),
    ('SegFormer',    'Transformer Encoder + MLP Dec.',  'Alto',   'Muy Alto','Datasets grandes, mejor que DeepLab'),
    ('SAM',          'ViT-H + Prompt Encoder',          'Muy Alto','SOTA',   'Zero-shot, sin fine-tuning'),
]

print(f'{"Arquitectura":<15} {"Componentes":<35} {"Params":<10} {"Precision":<12} {"Uso tipico"}')
print('-' * 100)
for row in arq_tabla:
    print(f'{row[0]:<15} {row[1]:<35} {row[2]:<10} {row[3]:<12} {row[4]}')

print()
print('=' * 100)

loss_tabla = [
    ('Cross-Entropy',   'Clases balanceadas, inicio de entrenamiento'),
    ('Weighted CE',     'Desbalance moderado, sustituye CE directo'),
    ('Dice Loss',       'Objetos pequeños, desbalance severo, imagenes medicas'),
    ('IoU Loss',        'Optimizar directamente la metrica de evaluacion'),
    ('Focal Loss',      'Desbalance + pixeles dificiles (bordes, oclusiones)'),
    ('CE + Dice',       'Opcion robusta general: estabilidad + precision de region'),
]

print(f'{"Perdida":<20} {"Cuándo usarla"}')
print('-' * 80)
for row in loss_tabla:
    print(f'{row[0]:<20} {row[1]}')

---

## Conclusiones

Este notebook recorrio los fundamentos de la segmentacion semantica con redes neuronales profundas:

**FCN**: primera arquitectura totalmente convolucional. Resuelve el problema de la salida densa, pero el upsampling bilineal desde una resolucion muy reducida limita la precision de los bordes.

**U-Net**: la arquitectura encoder-decoder con skip connections resuelve directamente la limitacion de la FCN. Las skip connections proveen detalle espacial fino que el bottleneck no puede recuperar por si solo. Es el estandar de facto en imagenes medicas y la base de la mayoria de las arquitecturas modernas.

**Funciones de perdida**: la eleccion impacta directamente en el rendimiento por clase. La Cross-Entropy naive falla con desbalance severo porque el gradiente esta dominado por el fondo. Dice Loss y la perdida combinada son mas robustas en ese escenario. En la practica, CE + Dice es la combinacion mas usada.

**Metricas**: la Pixel Accuracy es engañosa con desbalance. El **mIoU** es la metrica de referencia porque promedia el rendimiento por clase independientemente de su frecuencia. El Dice Coefficient es equivalente al F1 sobre regiones y es muy usado en el dominio medico.

**Augmentacion sincronizada**: toda transformacion geometrica debe aplicarse identicamente a la imagen y a la mascara. Las transformaciones de color o ruido solo se aplican a la imagen.

---

## Referencias

- Long, J., Shelhamer, E., & Darrell, T. (2015). Fully convolutional networks for semantic segmentation. *CVPR 2015*.
- Ronneberger, O., Fischer, P., & Brox, T. (2015). U-Net: Convolutional networks for biomedical image segmentation. *MICCAI 2015*.
- Chen, L.C., et al. (2018). Encoder-decoder with atrous separable convolution for semantic image segmentation (DeepLab v3+). *ECCV 2018*.
- Xie, E., et al. (2021). SegFormer: Simple and efficient design for semantic segmentation with transformers. *NeurIPS 2021*.
- Kirillov, A., et al. (2023). Segment Anything. *ICCV 2023*.
- Lin, T.Y., et al. (2017). Focal loss for dense object detection. *ICCV 2017*.
- Milletari, F., Navab, N., & Ahmadi, S.A. (2016). V-Net: Fully convolutional neural networks for volumetric medical image segmentation. *3DV 2016*.